Install ucirepo with pip install command

In [106]:
!pip install ucimlrepo


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Import the data and tools

In [107]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

1.
Load the CKD dataset into a single data frame and read its description.

In [108]:
#TODO: mitä "Load the CKD dataset into a single data frame and read its description." tarkoittaa???

# fetch dataset as a pandas DataFrame
chronic_kidney_disease :pd.DataFrame = fetch_ucirepo(id=336)

# data as pandas dataframes
measurement_data :pd.DataFrame = chronic_kidney_disease.data.features
diagnosis_data :pd.DataFrame = chronic_kidney_disease.data.targets

# metadata
metadata :pd.Series = chronic_kidney_disease.metadata


In [109]:
# metadata has descriptive information
print("\nMetadata:\n", metadata)


Metadata:
 {'uci_id': 336, 'name': 'Chronic Kidney Disease', 'repository_url': 'https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease', 'data_url': 'https://archive.ics.uci.edu/static/public/336/data.csv', 'abstract': 'This dataset can be used to predict the chronic kidney disease and it can be collected from the hospital nearly 2 months of period.', 'area': 'Other', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 400, 'num_features': 24, 'feature_types': ['Real'], 'demographics': ['Age'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2015, 'last_updated': 'Mon Mar 04 2024', 'dataset_doi': '10.24432/C5G020', 'creators': ['L. Rubini', 'P. Soundarapandian', 'P. Eswaran'], 'intro_paper': None, 'additional_info': {'summary': 'We use the following representation to collect the dataset\r\n                        age\t\t-\tage\t\r\n\t\t\tbp\t\t-\tblood pressure\r

In [110]:
# variable information
print("\nVariables:\n", chronic_kidney_disease.variables)


Variables:
      name     role         type demographic              description  \
0     age  Feature      Integer         Age                      NaN   
1      bp  Feature      Integer         NaN           blood pressure   
2      sg  Feature  Categorical         NaN         specific gravity   
3      al  Feature  Categorical         NaN                  albumin   
4      su  Feature  Categorical         NaN                    sugar   
5     rbc  Feature       Binary         NaN          red blood cells   
6      pc  Feature       Binary         NaN                 pus cell   
7     pcc  Feature       Binary         NaN          pus cell clumps   
8      ba  Feature       Binary         NaN                 bacteria   
9     bgr  Feature      Integer         NaN     blood glucose random   
10     bu  Feature      Integer         NaN               blood urea   
11     sc  Feature   Continuous         NaN         serum creatinine   
12    sod  Feature      Integer         NaN        

In [111]:
# pandas has .describe() method
print("\nvariables.describe:\n",chronic_kidney_disease.variables.describe())


variables.describe:
        name     role    type demographic     description   units  \
count    25       25      25           1              24      10   
unique   25        2       4           1              24       7   
top     age  Feature  Binary         Age  blood pressure  mgs/dl   
freq      1       24      11           1               1       3   

       missing_values  
count              25  
unique              2  
top               yes  
freq               24  


In [112]:
# data_features has data table
measurement_data.head(5)

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,15.4,44.0,7800.0,5.2,yes,yes,no,good,no,no
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,11.3,38.0,6000.0,NaN,no,no,no,good,no,no
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,9.6,31.0,7500.0,NaN,no,yes,no,poor,no,yes
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,11.2,32.0,6700.0,3.9,yes,no,no,poor,yes,yes
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,11.6,35.0,7300.0,4.6,no,no,no,good,no,no


In [113]:
# diagnosis_data has has diagnosis whether subject has chronic kidney disease (cdk) or no (nockd)
diagnosis_data

,class
0,ckd
1,ckd
2,ckd
3,ckd
4,ckd
...,...
395,notckd
396,notckd
397,notckd
398,notckd


2.
Construct a pipeline for modifying the data frame.

The modified data frame should meet the following requirements:

- It should include exactly the following columns:
  - age
  - blood pressure
  - specific gravity
  - albumin
  - sugar
  - blood glucose random
  - blood urea
  - sodium
  - potassium
  - hemoglobin
  - packed cell volume
  - white blood cell count
  - red blood cell count
  - class


In [114]:
# list of wanted columns
wanted_columns = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'class']

# abbreviations of columns
columns_dictionary = {
"age" : "age",
"bp" : "blood pressure",
"sg" : "specific gravity",
"al" : "albumin",
"su" : "sugar",
"rbc" : "red blood cells",
"pc" : "pus cell",
"pcc" : "pus cell clumps",
"ba" : "bacteria",
"bgr" : "blood glucose random",
"bu" : "blood urea",
"sc" : "serum creatinine",
"sod" : "sodium",
"pot" : "potassium",
"hemo" : "hemoglobin",
"pcv" : "packed cell volume",
"wbcc" : "white blood cell count",
"rbcc" : "red blood cell count",
"htn" : "hypertension",
"dm" : "diabetes mellitus",
"cad" : "coronary artery disease",
"appet" : "appetite",
"pe" : "pedal edema",
"ane" : "anemia",
"class" : "class"
}

# class column is in different table and has to be joined, make a new copy
modifed_data_frame :pd.DataFrame = measurement_data.join(diagnosis_data, how="left").copy(deep=True)
modifed_data_frame.columns

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'class'],
      dtype='str')

In [115]:
# filter wanted columns
modifed_data_frame :pd.DataFrame  = modifed_data_frame.filter(items=wanted_columns, axis="columns")
modifed_data_frame.head(5)

,age,bp,sg,al,su,bgr,bu,sod,pot,hemo,pcv,wbcc,rbcc,class
0,48.0,80.0,1.020,1.0,0.0,121.0,36.0,NaN,NaN,15.4,44.0,7800.0,5.2,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,18.0,NaN,NaN,11.3,38.0,6000.0,NaN,ckd
2,62.0,80.0,1.010,2.0,3.0,423.0,53.0,NaN,NaN,9.6,31.0,7500.0,NaN,ckd
3,48.0,70.0,1.005,4.0,0.0,117.0,56.0,111.0,2.5,11.2,32.0,6700.0,3.9,ckd
4,51.0,80.0,1.010,2.0,0.0,106.0,26.0,NaN,NaN,11.6,35.0,7300.0,4.6,ckd


- The hemoglobin values should be expressed in g/l. In the original data set, they are expressed as g/dl.





In [128]:
# first filter variables from original data frame to modified one by the name of the column
modifed_data_frame.variables =  chronic_kidney_disease.variables[chronic_kidney_disease.variables["name"].isin(wanted_columns)]

# test print
modifed_data_frame.variables

,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,NaN,year,yes
1,bp,Feature,Integer,NaN,blood pressure,mm/Hg,yes
2,sg,Feature,Categorical,NaN,specific gravity,NaN,yes
3,al,Feature,Categorical,NaN,albumin,NaN,yes
4,su,Feature,Categorical,NaN,sugar,NaN,yes
9,bgr,Feature,Integer,NaN,blood glucose random,mgs/dl,yes
10,bu,Feature,Integer,NaN,blood urea,mgs/dl,yes
12,sod,Feature,Integer,NaN,sodium,mEq/L,yes
13,pot,Feature,Continuous,NaN,potassium,mEq/L,yes
14,hemo,Feature,Continuous,NaN,hemoglobin,gms,yes


In [134]:
# change the unit of hemoglobin to g/l
modifed_data_frame.variables.replace("gms", "g/l", inplace=True)
#test print
modifed_data_frame.variables.iloc[9]

name                    hemo
role                 Feature
type              Continuous
demographic              NaN
description       hemoglobin
units                    g/l
missing_values           yes
Name: 14, dtype: str

- The values of the class column should be recoded as a or c (affected or control).

In [137]:
# see what is in class column
modifed_data_frame["class"].unique()

<StringArray>
['ckd', 'ckd\t', 'notckd']
Length: 3, dtype: str

In [138]:
# iterate over all values in class column and fix them TODO: tämä ei toimi
modifed_data_frame["class"].replace({"ckd" : "a", "ckd\t" : "a", "notckd" : "c"}, inplace=True)
modifed_data_frame["class"].unique()


C:\Users\valas\AppData\Local\Temp\ipykernel_15124\2155903976.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  modifed_data_frame["class"].replace({"ckd" : "a", "ckd\t" : "a", "notckd" : "c"}, inplace=True)


<StringArray>
['ckd', 'ckd\t', 'notckd']
Length: 3, dtype: str

- Rows with three or more missing values should be removed. Indicate the number of rows left in the modified data frame.

3.
Next, split the data frame into two data frames, one for the affected individuals and one for the control individuals. Display the data frames, and indicate the number of rows in each data frame.

4.
For each data frame, calculate the basic statistics for each column, and provide clear, readable histograms for each numerical column. Do you see any outliers? If so, how would you handle them?




5.
Finally, calculate the correlation matrix and visualize it for each data frame. Clearly describe the results and your interpretation for it.